In [3]:
import geopandas as gpd
from pathlib import Path

In [4]:
#Locate your GeoJSON file
project_dir = Path.cwd().parent

geography_dir = project_dir / "data" / "raw" / "geography"

list(geography_dir.iterdir())

[WindowsPath('c:/Users/Mkosh/OneDrive/Desktop/Project/AgriRain-ML/data/raw/geography/district_nwic_geojson.zip')]

In [5]:
#Inspect the ZIP archive
import zipfile

zip_path = geography_dir / "district_nwic_geojson.zip"

with zipfile.ZipFile(zip_path, "r") as archive:
    print(archive.namelist())

['district_nwic.GeoJSON']


In [6]:
#Extract the GeoJSON
with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(geography_dir)

geojson_path = geography_dir / "district_nwic.GeoJSON"

print("File exists:", geojson_path.exists())

File exists: True


In [8]:
#Load GeoJSON without GDAL
import json

with open(geojson_path, "r", encoding="utf-8") as file:
    geojson_data = json.load(file)

print("GeoJSON loaded successfully!")
print("Number of features:", len(geojson_data["features"]))

GeoJSON loaded successfully!
Number of features: 733


In [9]:
#Inspect the GeoJSON structure
geojson_data.keys()

dict_keys(['type', 'name', 'crs', 'features'])

In [10]:
#Inspect the CRS
geojson_data["crs"]

{'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:EPSG::7755'}}

In [11]:
#Inspect one geographical feature
geojson_data["features"][0]

{'type': 'Feature',
 'properties': {'id': 377,
  'objectid': 377,
  'state': 'MN',
  'stcode': '14',
  'district': 'Kamjong',
  'dtcode': '717',
  'shape_length': 230554.29191157038,
  'shape_area': 2225794048.57499,
  'ds_name': ' Administrative Boundary',
  'src_agency': 'Survey of India (SOI)',
  'state_name': 'Manipur'},
 'geometry': {'type': 'MultiPolygon',
  'coordinates': [[[[5454524.1995334625, 4176938.537444787],
     [5454476.365533482, 4176888.5985429296],
     [5454296.537830564, 4176861.3672390194],
     [5453988.1403246755, 4176861.1577332052],
     [5453778.159520192, 4176886.393229728],
     [5453727.075619096, 4176892.5326288864],
     [5453685.8024184, 4176887.782128011],
     [5453653.102818409, 4176853.96042675],
     [5453667.703019768, 4176796.810725941],
     [5453681.644121056, 4176742.2408251576],
     [5453662.860324042, 4176564.8215214126],
     [5453685.955427382, 4176411.0365189095],
     [5453651.068429127, 4176282.945415806],
     [5453631.326131703, 4176

In [12]:
#Inspect all geometry types
from collections import Counter

geometry_types = Counter(
    feature["geometry"]["type"]
    for feature in geojson_data["features"]
)

geometry_types

Counter({'MultiPolygon': 733})

In [14]:
import pandas as pd

In [15]:
#Create a table of district attributes
properties = [
    feature["properties"]
    for feature in geojson_data["features"]
]

properties_df = pd.DataFrame(properties)

In [16]:
properties_df.isnull().sum()

id              0
objectid        0
state           0
stcode          0
district        0
dtcode          0
shape_length    0
shape_area      0
ds_name         0
src_agency      0
state_name      0
dtype: int64

In [17]:
#Check duplicate district identifiers
properties_df["dtcode"].duplicated().sum()

np.int64(1)

In [18]:
#Locate the duplicate
properties_df[
    properties_df["dtcode"].duplicated(keep=False)
][["state_name", "district", "stcode", "dtcode"]]

,state_name,district,stcode,dtcode
730,Jammu & Kashmir,Mirpur,01,999
731,Jammu & Kashmir,Muzaffarabad,01,999


In [19]:
#Check geometry validity
from shapely.geometry import shape

invalid_geometries = sum(
    not shape(feature["geometry"]).is_valid
    for feature in geojson_data["features"]
)

print("Invalid geometries:", invalid_geometries)

Invalid geometries: 13


In [20]:
#Locate invalid districts
invalid_districts = [
    feature["properties"]
    for feature in geojson_data["features"]
    if not shape(feature["geometry"]).is_valid
]

invalid_df = pd.DataFrame(invalid_districts)

invalid_df[["state_name", "district", "dtcode"]]

,state_name,district,dtcode
0,Andhra Pradesh,Srikakulam,519
1,Andhra Pradesh,Vizianagaram,521
2,Gujarat,Kachchh,449
3,Gujarat,Patan,455
4,Madhya Pradesh,Agar Malwa,667
5,Odisha,Gajapati,353
6,Odisha,Nuapada,368
7,Tamil Nadu,Thanjavur,586
8,Tamil Nadu,Cuddalore,570
9,Tripura,Gomati,654


In [21]:
#Identify what's wrong
from shapely.validation import explain_validity

invalid_reasons = [
    {
        "state": feature["properties"]["state_name"],
        "district": feature["properties"]["district"],
        "reason": explain_validity(shape(feature["geometry"]))
    }
    for feature in geojson_data["features"]
    if not shape(feature["geometry"]).is_valid
]

invalid_reasons_df = pd.DataFrame(invalid_reasons)

invalid_reasons_df


,state,district,reason
0,Andhra Pradesh,Srikakulam,Ring Self-intersection[4421062.05456903 344072...
1,Andhra Pradesh,Vizianagaram,Ring Self-intersection[4353398.66051877 346100...
2,Gujarat,Kachchh,Ring Self-intersection[3118153.06280184 393809...
3,Gujarat,Patan,Ring Self-intersection[3211298.12301693 397883...
4,Madhya Pradesh,Agar Malwa,Ring Self-intersection[3584686.14969175 397703...
5,Odisha,Gajapati,Ring Self-intersection[4443915.43143389 352517...
6,Odisha,Nuapada,Ring Self-intersection[4292071.6822443 3599746...
7,Tamil Nadu,Thanjavur,Ring Self-intersection[3906687.40531381 252434...
8,Tamil Nadu,Cuddalore,Ring Self-intersection[3966646.19507602 267279...
9,Tripura,Gomati,Ring Self-intersection[5182719.93175986 399513...


In [22]:
#Check for empty geometries
empty_geometries = sum(
    shape(feature["geometry"]).is_empty
    for feature in geojson_data["features"]
)

print("Empty geometries:", empty_geometries)

Empty geometries: 0


In [23]:
#Check coordinate bounds
all_geometries = [
    shape(feature["geometry"])
    for feature in geojson_data["features"]
]

geometry_bounds = [geometry.bounds for geometry in all_geometries]

print("Overall bounds:", (
    min(bounds[0] for bounds in geometry_bounds),
    min(bounds[1] for bounds in geometry_bounds),
    max(bounds[2] for bounds in geometry_bounds),
    max(bounds[3] for bounds in geometry_bounds)
))

Overall bounds: (2818369.546553203, 2177948.4534365367, 5679118.542969628, 5444563.235709688)


In [24]:
#Confirm CSR
geojson_data["crs"]

{'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:EPSG::7755'}}

In [26]:
#Verify EPSG:7755
from pyproj import CRS

crs = CRS.from_epsg(7755)

print("CRS Name:", crs.name)
print("Is Projected:", crs.is_projected)
print("Coordinate Axes:", crs.axis_info)

CRS Name: WGS 84 / India NSF LCC
Is Projected: True
Coordinate Axes: [Axis(name=Easting, abbrev=X, direction=east, unit_auth_code=EPSG, unit_code=9001, unit_name=metre), Axis(name=Northing, abbrev=Y, direction=north, unit_auth_code=EPSG, unit_code=9001, unit_name=metre)]


In [ ]:
#Geography CRS Verification
#- CRS: EPSG:7755
#- Full name: WGS 84 / India NSF LCC
#- Projection: Lambert Conformal Conic
#- Coordinate type: Projected
#- Units: Metres (Easting and Northing)
#- Verification: Successful using PyProj.

In [27]:
#Investigate invalid geometries
from shapely.geometry import shape
from shapely.validation import explain_validity

invalid_districts = []

for feature in geojson_data["features"]:
    geometry = shape(feature["geometry"])

    if not geometry.is_valid:
        invalid_districts.append({
            "State": feature["properties"]["state_name"],
            "District": feature["properties"]["district"],
            "Reason": explain_validity(geometry)
        })

invalid_districts_df = pd.DataFrame(invalid_districts)

invalid_districts_df

,State,District,Reason
0,Andhra Pradesh,Srikakulam,Ring Self-intersection[4421062.05456903 344072...
1,Andhra Pradesh,Vizianagaram,Ring Self-intersection[4353398.66051877 346100...
2,Gujarat,Kachchh,Ring Self-intersection[3118153.06280184 393809...
3,Gujarat,Patan,Ring Self-intersection[3211298.12301693 397883...
4,Madhya Pradesh,Agar Malwa,Ring Self-intersection[3584686.14969175 397703...
5,Odisha,Gajapati,Ring Self-intersection[4443915.43143389 352517...
6,Odisha,Nuapada,Ring Self-intersection[4292071.6822443 3599746...
7,Tamil Nadu,Thanjavur,Ring Self-intersection[3906687.40531381 252434...
8,Tamil Nadu,Cuddalore,Ring Self-intersection[3966646.19507602 267279...
9,Tripura,Gomati,Ring Self-intersection[5182719.93175986 399513...


In [28]:
#Repair one geometry as a test
from shapely import make_valid

srikakulam_feature = next(
    feature for feature in geojson_data["features"]
    if feature["properties"]["district"] == "Srikakulam"
)

original_geometry = shape(srikakulam_feature["geometry"])

repaired_geometry = make_valid(original_geometry)

print("Original valid:", original_geometry.is_valid)
print("Repaired valid:", repaired_geometry.is_valid)

print("Original geometry type:", original_geometry.geom_type)
print("Repaired geometry type:", repaired_geometry.geom_type)

print("Original area:", original_geometry.area)
print("Repaired area:", repaired_geometry.area)

Original valid: False
Repaired valid: True
Original geometry type: MultiPolygon
Repaired geometry type: Polygon
Original area: 5697773756.131599
Repaired area: 5697773756.131628


In [ ]:
#- Method: Shapely make_valid().
#- Original validity: Invalid.
#- Repaired validity: Valid.
#- Original geometry: MultiPolygon.
#- Repaired geometry: Polygon.
#- Area difference: Approximately 0.000029 m².
#- Conclusion: Repair successful with negligible area change.

In [29]:
#Test all 13 geometries
repaired_features = []

for feature in geojson_data["features"]:
    geometry = shape(feature["geometry"])

    if not geometry.is_valid:
        geometry = make_valid(geometry)

    repaired_features.append({
        "properties": feature["properties"],
        "geometry": geometry
    })

print("Total districts:", len(repaired_features))

print(
    "Remaining invalid geometries:",
    sum(not feature["geometry"].is_valid
        for feature in repaired_features)
)

Total districts: 733
Remaining invalid geometries: 0


In [30]:
#Investigate geometry types
from collections import Counter

geometry_types_after = Counter(
    feature["geometry"].geom_type
    for feature in repaired_features
)

geometry_types_after

Counter({'MultiPolygon': 725, 'Polygon': 8})

In [31]:
#Which districts changed?
changed_geometry_types = []

for original, repaired in zip(
    geojson_data["features"],
    repaired_features
):
    original_type = original["geometry"]["type"]
    repaired_type = repaired["geometry"].geom_type

    if original_type != repaired_type:
        changed_geometry_types.append({
            "State": original["properties"]["state_name"],
            "District": original["properties"]["district"],
            "Original": original_type,
            "Repaired": repaired_type
        })

pd.DataFrame(changed_geometry_types)

,State,District,Original,Repaired
0,Andhra Pradesh,Srikakulam,MultiPolygon,Polygon
1,Gujarat,Kachchh,MultiPolygon,Polygon
2,Madhya Pradesh,Agar Malwa,MultiPolygon,Polygon
3,Odisha,Nuapada,MultiPolygon,Polygon
4,Tamil Nadu,Thanjavur,MultiPolygon,Polygon
5,Tripura,Gomati,MultiPolygon,Polygon
6,Uttar Pradesh,Azamgarh,MultiPolygon,Polygon
7,Uttar Pradesh,Etah,MultiPolygon,Polygon


In [32]:
#Compare the area changes across all 13 districts
area_comparison = []

for original, repaired in zip(
    geojson_data["features"],
    repaired_features
):
    original_geometry = shape(original["geometry"])
    repaired_geometry = repaired["geometry"]

    if not original_geometry.is_valid:
        area_difference = (
            repaired_geometry.area - original_geometry.area
        )

        percentage_change = (
            abs(area_difference) / original_geometry.area
        ) * 100

        area_comparison.append({
            "State": original["properties"]["state_name"],
            "District": original["properties"]["district"],
            "Original_Area": original_geometry.area,
            "Repaired_Area": repaired_geometry.area,
            "Percentage_Change": percentage_change
        })

area_comparison_df = pd.DataFrame(area_comparison)

area_comparison_df.sort_values(
    "Percentage_Change",
    ascending=False
)

,State,District,Original_Area,Repaired_Area,Percentage_Change
0,Andhra Pradesh,Srikakulam,5.697774e+09,5.697774e+09,5.021300e-13
5,Odisha,Gajapati,3.962683e+09,3.962683e+09,3.850620e-13
4,Madhya Pradesh,Agar Malwa,2.628619e+09,2.628619e+09,3.083837e-13
10,Uttar Pradesh,Azamgarh,4.057577e+09,4.057577e+09,2.937943e-13
11,Uttar Pradesh,Etah,2.338703e+09,2.338703e+09,2.854454e-13
8,Tamil Nadu,Cuddalore,3.734423e+09,3.734423e+09,2.170679e-13
9,Tripura,Gomati,1.597285e+09,1.597285e+09,2.089709e-13
12,Uttar Pradesh,Muzaffarnagar,2.656207e+09,2.656207e+09,1.795181e-13
3,Gujarat,Patan,5.551721e+09,5.551721e+09,1.546020e-13
6,Odisha,Nuapada,3.718207e+09,3.718207e+09,1.538926e-13


In [37]:
properties_df[
    properties_df["dtcode"] == "999"
][[
    "id",
    "objectid",
    "state_name",
    "district",
    "stcode",
    "dtcode"
]]

,id,objectid,state_name,district,stcode,dtcode
730,731,731,Jammu & Kashmir,Mirpur,01,999
731,732,732,Jammu & Kashmir,Muzaffarabad,01,999


In [38]:
#Final geography quality check
unexpected_types = [
    feature["geometry"].geom_type
    for feature in repaired_features
    if feature["geometry"].geom_type not in ["Polygon", "MultiPolygon"]
]

print("Unexpected geometry types:", len(unexpected_types))

Unexpected geometry types: 0
